# Steel Mechanical Properties ML

End-to-end analysis of the 435-sample supplementary steel dataset. The workflow covers EDA, SciPy testing, correlation/VIF, leakage-safe ML, feature selection, ensemble comparisons, repeated cross-validation, held-out testing, SHAP interpretation, residual analysis, and final summary.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/steel_mechanical_properties_435.csv')
print('Dataset shape:', df.shape)
display(df.head())

feature_cols = ['Al','AlS','C','Cr','Cu','Mn','Mo','N','Nb','Ni','P','S','Si','Ti','V','FRT','CTT']
target_cols = ['YS','UTS','EL']


In [ ]:
print('Missing values:')
display(df.isnull().sum().to_frame('Missing Count'))
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate Sample_IDs:', df['Sample_ID'].duplicated().sum())
display(df.describe().T)


In [ ]:
from scipy import stats
for target in target_cols:
    statistic, p_value = stats.shapiro(df[target])
    print(f'{target}: statistic={statistic:.4f}, p-value={p_value:.6f}')


In [ ]:
# Feature cardinality and rare process levels
display(df[feature_cols].nunique().sort_values().to_frame('Unique Values'))
print('CTT distribution:')
display(df['CTT'].value_counts().sort_index().to_frame('Count'))
print('CTT=650 observations:')
display(df[df['CTT']==650])


In [ ]:
# IQR screening: observations are flagged, not deleted
analysis_cols = feature_cols + target_cols
rows=[]
for col in analysis_cols:
    q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1
    rows.append({'Feature':col,'Q1':q1,'Q3':q3,'IQR':iqr,'Outlier Count':((df[col] < q1-1.5*iqr)|(df[col] > q3+1.5*iqr)).sum()})
display(pd.DataFrame(rows).sort_values('Outlier Count',ascending=False))


In [ ]:
# Pearson correlations
corr = df[feature_cols+target_cols].corr()
plt.figure(figsize=(16,12)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0); plt.tight_layout(); plt.show()


In [ ]:
from scipy.stats import pearsonr, spearmanr
corr_results=[]
for f in feature_cols:
    for t in target_cols:
        r,p=pearsonr(df[f],df[t]); corr_results.append({'Feature':f,'Target':t,'Pearson_r':r,'p_value':p})
pearson_results=pd.DataFrame(corr_results)
display(pearson_results.sort_values(['Target','Pearson_r'],ascending=[True,False]).round(4))


In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split
X=df[feature_cols].copy(); y=df[target_cols].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)
print('CTT train/test:'); print(X_train['CTT'].value_counts().sort_index()); print(X_test['CTT'].value_counts().sort_index())


In [ ]:
# Leakage-safe preprocessing and baseline models
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
from sklearn.base import clone
import numpy as np
cv=KFold(n_splits=5,shuffle=True,random_state=42)
models={
 'Ridge + StandardScaler':Pipeline([('scaler',StandardScaler()),('model',Ridge(alpha=1.0))]),
 'Ridge + MinMaxScaler':Pipeline([('scaler',MinMaxScaler()),('model',Ridge(alpha=1.0))]),
 'SVR + StandardScaler':Pipeline([('scaler',StandardScaler()),('model',MultiOutputRegressor(SVR(kernel='rbf',C=10,epsilon=.1)))]),
 'Random Forest':RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1),
 'Extra Trees':ExtraTreesRegressor(n_estimators=300,random_state=42,n_jobs=-1),
 'Gradient Boosting':MultiOutputRegressor(GradientBoostingRegressor(n_estimators=200,learning_rate=.05,max_depth=3,random_state=42))}
results=[]
for name,model in models.items():
    fm={t:{'R2':[],'RMSE':[],'MAE':[]} for t in target_cols}
    for tr,va in cv.split(X_train):
        m=clone(model); m.fit(X_train.iloc[tr],y_train.iloc[tr]); pred=m.predict(X_train.iloc[va])
        for j,t in enumerate(target_cols):
            fm[t]['R2'].append(r2_score(y_train.iloc[va][t],pred[:,j])); fm[t]['RMSE'].append(np.sqrt(mean_squared_error(y_train.iloc[va][t],pred[:,j]))); fm[t]['MAE'].append(mean_absolute_error(y_train.iloc[va][t],pred[:,j]))
    for t in target_cols: results.append({'Model':name,'Target':t,'R2_mean':np.mean(fm[t]['R2']),'R2_std':np.std(fm[t]['R2']),'RMSE_mean':np.mean(fm[t]['RMSE']),'MAE_mean':np.mean(fm[t]['MAE'])})
baseline_results=pd.DataFrame(results); display(baseline_results.round(4))


In [ ]:
# Mutual-information feature selection
from sklearn.feature_selection import SelectKBest, mutual_info_regression
mi_tables={}
for target in target_cols:
    mi=mutual_info_regression(X_train,y_train[target],random_state=42)
    tab=pd.DataFrame({'Feature':feature_cols,'MI':mi}).sort_values('MI',ascending=False)
    mi_tables[target]=tab; print(target); display(tab.round(4))


In [ ]:
# MI-selected Random Forest comparison
mi_rf_results=[]
for target in target_cols:
    for k in [5,8,12,17]:
        r2s=[]; rmses=[]; maes=[]
        for tr,va in cv.split(X_train):
            sel=SelectKBest(mutual_info_regression,k=k); Xt=sel.fit_transform(X_train.iloc[tr],y_train[target].iloc[tr]); Xv=sel.transform(X_train.iloc[va])
            m=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1); m.fit(Xt,y_train[target].iloc[tr]); p=m.predict(Xv)
            r2s.append(r2_score(y_train[target].iloc[va],p)); rmses.append(np.sqrt(mean_squared_error(y_train[target].iloc[va],p))); maes.append(mean_absolute_error(y_train[target].iloc[va],p))
        mi_rf_results.append({'Target':target,'Top_k':k,'R2_mean':np.mean(r2s),'R2_std':np.std(r2s),'RMSE_mean':np.mean(rmses),'MAE_mean':np.mean(maes)})
display(pd.DataFrame(mi_rf_results).round(4))


In [ ]:
# XGBoost and LightGBM comparisons
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
def boosted_cv(ModelClass, kwargs):
    out=[]
    for target in target_cols:
        vals=[]
        for tr,va in cv.split(X_train):
            sel=SelectKBest(mutual_info_regression,k=12); Xt=sel.fit_transform(X_train.iloc[tr],y_train[target].iloc[tr]); Xv=sel.transform(X_train.iloc[va])
            m=ModelClass(**kwargs); m.fit(Xt,y_train[target].iloc[tr]); p=m.predict(Xv)
            vals.append([r2_score(y_train[target].iloc[va],p),np.sqrt(mean_squared_error(y_train[target].iloc[va],p)),mean_absolute_error(y_train[target].iloc[va],p)])
        a=np.array(vals); out.append({'Target':target,'R2_mean':a[:,0].mean(),'R2_std':a[:,0].std(),'RMSE_mean':a[:,1].mean(),'MAE_mean':a[:,2].mean()})
    return pd.DataFrame(out)
xgb_summary=boosted_cv(XGBRegressor,dict(n_estimators=500,max_depth=4,learning_rate=.03,subsample=.85,colsample_bytree=.85,reg_alpha=.05,reg_lambda=1.0,objective='reg:squarederror',random_state=42,n_jobs=-1)); print('XGBoost'); display(xgb_summary.round(4))
lgb_summary=boosted_cv(LGBMRegressor,dict(n_estimators=500,learning_rate=.03,max_depth=4,num_leaves=15,subsample=.85,colsample_bytree=.85,reg_alpha=.05,reg_lambda=1.0,random_state=42,verbosity=-1,n_jobs=-1)); print('LightGBM'); display(lgb_summary.round(4))


In [ ]:
# Stacking comparison
stack_results=[]
for target in target_cols:
    stack=StackingRegressor(estimators=[('rf',RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)),('et',ExtraTreesRegressor(n_estimators=300,random_state=42,n_jobs=-1)),('gb',GradientBoostingRegressor(n_estimators=200,learning_rate=.05,max_depth=3,random_state=42)),('svr',Pipeline([('scaler',StandardScaler()),('model',SVR(C=10,epsilon=.1,kernel='rbf'))]))],final_estimator=Ridge(alpha=1.0),cv=5,n_jobs=-1)
    vals=[]
    for tr,va in cv.split(X_train):
        stack.fit(X_train.iloc[tr],y_train[target].iloc[tr]); p=stack.predict(X_train.iloc[va]); vals.append([r2_score(y_train[target].iloc[va],p),np.sqrt(mean_squared_error(y_train[target].iloc[va],p)),mean_absolute_error(y_train[target].iloc[va],p)])
    a=np.array(vals); stack_results.append({'Target':target,'R2_mean':a[:,0].mean(),'R2_std':a[:,0].std(),'RMSE_mean':a[:,1].mean(),'MAE_mean':a[:,2].mean()})
display(pd.DataFrame(stack_results).round(4))


In [ ]:
# Repeated 5-fold CV for final RF configuration
from sklearn.model_selection import RepeatedKFold
repeated_cv=RepeatedKFold(n_splits=5,n_repeats=5,random_state=42)
repeated_results=[]; best_k={'YS':12,'UTS':12,'EL':8}
for target in target_cols:
    r2s=[]; rmses=[]; maes=[]
    for tr,va in repeated_cv.split(X_train):
        sel=SelectKBest(mutual_info_regression,k=best_k[target]); Xt=sel.fit_transform(X_train.iloc[tr],y_train[target].iloc[tr]); Xv=sel.transform(X_train.iloc[va])
        m=RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1); m.fit(Xt,y_train[target].iloc[tr]); p=m.predict(Xv)
        r2s.append(r2_score(y_train[target].iloc[va],p)); rmses.append(np.sqrt(mean_squared_error(y_train[target].iloc[va],p))); maes.append(mean_absolute_error(y_train[target].iloc[va],p))
    repeated_results.append({'Target':target,'R2_mean':np.mean(r2s),'R2_std':np.std(r2s),'R2_min':np.min(r2s),'R2_max':np.max(r2s),'RMSE_mean':np.mean(rmses),'MAE_mean':np.mean(maes)})
repeated_results=pd.DataFrame(repeated_results); display(repeated_results.round(4))


In [ ]:
# Final models and untouched test evaluation
final_rf_models={}; final_predictions={}; final_selectors={}
for target in target_cols:
    sel=SelectKBest(mutual_info_regression,k=best_k[target]); Xt=sel.fit_transform(X_train,y_train[target]); Xte=sel.transform(X_test)
    m=RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1); m.fit(Xt,y_train[target]); pred=m.predict(Xte)
    final_selectors[target]=sel; final_rf_models[target]=m; final_predictions[target]=pred
    print(target, list(np.array(feature_cols)[sel.get_support()]))

final_results=[]
for target in target_cols:
    pred=final_predictions[target]
    final_results.append({'Target':target,'R2_test':r2_score(y_test[target],pred),'RMSE_test':np.sqrt(mean_squared_error(y_test[target],pred)),'MAE_test':mean_absolute_error(y_test[target],pred)})
final_results=pd.DataFrame(final_results); display(final_results.round(4))


In [ ]:
# SHAP interpretation
import shap
shap_values_all={}
for target in target_cols:
    sel=final_selectors[target]; Xte=sel.transform(X_test); names=np.array(feature_cols)[sel.get_support()]
    explainer=shap.TreeExplainer(final_rf_models[target]); sv=explainer.shap_values(Xte)
    shap_values_all[target]={'values':sv,'features':names,'X':Xte}
    shap.summary_plot(sv,Xte,feature_names=names,show=True)


In [ ]:
# Actual vs predicted and residual plots
fig,axes=plt.subplots(1,3,figsize=(18,5))
for i,target in enumerate(target_cols):
    actual=y_test[target].values; pred=final_predictions[target]; lo=min(actual.min(),pred.min()); hi=max(actual.max(),pred.max())
    axes[i].scatter(actual,pred,alpha=.7); axes[i].plot([lo,hi],[lo,hi],linestyle='--'); axes[i].set_xlabel('Actual'); axes[i].set_ylabel('Predicted'); axes[i].set_title(f'{target}: Actual vs Predicted')
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(1,3,figsize=(18,5))
for i,target in enumerate(target_cols):
    pred=final_predictions[target]; residuals=y_test[target].values-pred
    axes[i].scatter(pred,residuals,alpha=.7); axes[i].axhline(0,linestyle='--'); axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('Residual'); axes[i].set_title(f'{target}: Residual Plot')
plt.tight_layout(); plt.show()


# Final Project Summary

The supplementary dataset contained 435 samples, 17 numerical input variables, and three targets: YS, UTS and EL. No missing values or duplicate records were identified. Multiple regression, kernel, tree, boosting and stacking models were evaluated with leakage-safe preprocessing and target-specific mutual-information feature selection.

The final Random Forest configurations used 12 selected features for YS and UTS and 8 for EL. Repeated 5-fold CV produced R² values of 0.437 ± 0.140 (YS), 0.566 ± 0.138 (UTS), and 0.390 ± 0.111 (EL). On the untouched test split, R² was 0.141 (YS), 0.234 (UTS), and 0.549 (EL).

The dataset does not support an honest 0.85–0.95 held-out R² with the available predictors. The study therefore reports leakage-safe validation rather than tuning against the test set to inflate performance. SHAP analysis identified Cr, C and Mn as important contributors for YS; C, N, Cr and Mn for UTS; and Cr, Al, N, AlS, P and C for EL.
